0      9222
1      7763
2      7764
3      7765
4      7766
       ... 
227    9461
228    9656
229    9657
230    9658
231    9662
Name: session_key, Length: 232, dtype: int64

In [10]:
from tqdm import tqdm
import numpy as np
# F1 Data Export
# This notebook connects to QuestDB and exports data from all tables for each session key.

import sys
import json
from pathlib import Path
import pandas as pd

# Add the parent directory to sys.path to import the client
sys.path.append('..')
from backend_python.db.client import QuestDBClient

# Initialize database connection
client = QuestDBClient()


all_sessions = client.query("""
select 
    * 

from sessions
""")['session_key']

all_sessions


# Get all tables with session_key from schema
tables_with_session = [
    # 'car_data',
    'drivers', 
    'intervals',
    'laps',
    # 'location',
    'pit',
    'position', 
    'race_control',
    'stints',
    'team_radio',
    'weather'
]

sizes = []

# Loop through each session with progress bar
for session_key in tqdm(all_sessions, desc="Processing sessions"):
    # Dictionary to store data for this session
    session_data = {}
    
    # Fetch data from each table for this session
    for table in tables_with_session:
        # print(table)
        data = client.query(f"""
            SELECT *
            FROM {table}
            WHERE session_key = {session_key}
        """)
        
        if not data.empty:
            # Convert timestamps to strings before serializing
            for col in data.select_dtypes(include=['datetime64[ns]']).columns:
                data[col] = data[col].astype(str)
            
            session_data[table] = data.to_dict('records')
    
    # Export to JSON if we have data
    if session_data:
        json_path = Path(f'temp_session_{session_key}.json')
        
        with open(json_path, 'w') as f:
            json.dump(session_data, f)
            
        # Get file size in MB
        size_mb = json_path.stat().st_size / (1024 * 1024)
        
        # Delete temp file
        json_path.unlink()
        
        sizes.append([session_key, size_mb])
        
# Create DataFrame of sizes
sizes_df = pd.DataFrame(sizes, columns=['session_key', 'size_mb'])

# Display summary statistics
print("\nSize statistics (MB):")
print(sizes_df['size_mb'].describe())




In [19]:
data = client.query(f"""
            SELECT 
                *
            FROM car_data c 
            inner join sessions se on se.session_key = c.session_key
            WHERE se.session_key = 7953 and se.session_name = 'Race'
        """)#.drop(columns=['timestamp', 'session_key'])

# Convert timestamps to strings before serializing
for col in data.select_dtypes(include=['datetime64[ns]']).columns:
    data[col] = data[col].astype(str)

# Save to temp JSON file
json_path = Path('temp_car_data.json')
with open(json_path, 'w') as f:
    json.dump(data.to_dict('records'), f)

# Get and print file size in MB
size_mb = json_path.stat().st_size / (1024 * 1024)
print(f"File size: {size_mb:.2f} MB")

# Delete temp file
json_path.unlink()

File size: 265.70 MB


In [13]:
client.query(f"""
            select 
                * 
            from meetings 
            where session_key = 1140
        """)

Query error: (psycopg2.DatabaseError) table does not exist [table=metings]
LINE 4:             from metings 
                         ^

[SQL: 
            select 
                * 
            from metings 
            where meeting_key = 1140
        ]
(Background on this error at: https://sqlalche.me/e/20/4xp6)


""
